# 05.16 - Model Interpretation

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

How do we explain what a model learned and why it makes specific predictions? We cover feature importance, permutation importance, and partial dependence.

## 2. Why Does This Matter?

Interpretability builds trust, helps debugging, and is required in regulated domains (finance, healthcare).

## 3. Prerequisites

- Unit 05.6 (Random Forests), Unit 05.7 (Gradient Boosting)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Use feature importance
- Use permutation importance
- Use partial dependence plots
- Explain individual predictions

## 5. Mental Model

- **Feature importance**: how much each feature reduces impurity.
- **Permutation importance**: how much accuracy drops when a feature is shuffled.
- **Partial dependence**: how predictions change as a feature varies.

These help explain global model behavior.


## 6. Generate Data

Create a dataset with known feature importance.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, n_informative=4, n_redundant=3, n_repeated=0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print(f"Test accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")


Test accuracy: 0.917


## 7. Feature Importance

Random forest feature importance.


In [2]:
importances = model.feature_importances_
print("Feature importance:")
for i, imp in enumerate(importances):
    print(f"  Feature {i}: {imp:.3f}")
print("\nThe first 4 features (informative) should rank highest.")


Feature importance:
  Feature 0: 0.169
  Feature 1: 0.125
  Feature 2: 0.166
  Feature 3: 0.079
  Feature 4: 0.017
  Feature 5: 0.095
  Feature 6: 0.019
  Feature 7: 0.018
  Feature 8: 0.175
  Feature 9: 0.139

The first 4 features (informative) should rank highest.


## 8. Permutation Importance

Shuffle each feature and measure the drop in accuracy.


In [3]:
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
print("Permutation importance:")
for i in range(len(result.importances_mean)):
    print(f"  Feature {i}: {result.importances_mean[i]:.4f} +/- {result.importances_std[i]:.4f}")
print("\nHigher = more important (shuffling it hurts accuracy more).")


Permutation importance:
  Feature 0: 0.1537 +/- 0.0170
  Feature 1: 0.0417 +/- 0.0107
  Feature 2: 0.0193 +/- 0.0111
  Feature 3: 0.0140 +/- 0.0071
  Feature 4: -0.0037 +/- 0.0010
  Feature 5: 0.0100 +/- 0.0076
  Feature 6: -0.0007 +/- 0.0020
  Feature 7: -0.0000 +/- 0.0026
  Feature 8: 0.0747 +/- 0.0101
  Feature 9: 0.0563 +/- 0.0147

Higher = more important (shuffling it hurts accuracy more).


## 9. Partial Dependence

How does the prediction change as a single feature varies?


In [4]:
fig, ax = plt.subplots(figsize=(8, 4))
PartialDependenceDisplay.from_estimator(model, X_train, [0, 1], ax=ax)
plt.suptitle("Partial dependence of features 0 and 1")
plt.tight_layout()
plt.show()
print("Partial dependence shows the average effect of each feature.")


Partial dependence shows the average effect of each feature.


C:\Users\PC\AppData\Local\Temp\ipykernel_20880\3952586850.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Interpreting a Single Prediction

For a single tree, we can trace the path. For forests, we use aggregate importance.


In [5]:
# Show the decision path for a single sample
sample = X_test[0].reshape(1, -1)
pred = model.predict(sample)[0]
print(f"Sample prediction: class {pred}")
print("\nFor a random forest, we can't trace a single path easily.")
print("But feature importance tells us which features drive predictions.")


Sample prediction: class 1

For a random forest, we can't trace a single path easily.
But feature importance tells us which features drive predictions.


## 11. Failure Case: Misleading Importance

Feature importance can be biased toward high-cardinality features.


In [6]:
print("Feature importance caveats:")
print("  - Favors high-cardinality features.")
print("  - Correlated features split importance.")
print("  - Permutation importance is more robust.")
print("\nUse multiple interpretation methods together.")


Feature importance caveats:
  - Favors high-cardinality features.
  - Correlated features split importance.
  - Permutation importance is more robust.

Use multiple interpretation methods together.


## 12. Debugging: Common Errors

- **Trusting importance blindly**: check with permutation.
- **Correlated features**: importance is split.
- **Not scaling for linear models**: coefficients need scaling.

## 13. Real-World Considerations

- Use permutation importance for robustness.
- Partial dependence shows direction of effect.
- For regulated domains, prefer interpretable models.

## 14. Common Mistakes

- Over-interpreting feature importance.
- Ignoring correlated features.

## 15. When NOT to Use

- When you need exact per-prediction explanations (use SHAP/LIME).

## 16. Challenge

Compare feature importance and permutation importance and identify which features are truly important.


In [7]:
# Challenge: compare importance methods
print(f"{'Feature':<10} {'Tree imp':<10} {'Perm imp':<10}")
print("-" * 32)
for i in range(10):
    print(f"{i:<10} {importances[i]:<10.3f} {result.importances_mean[i]:<10.4f}")
print("\nBoth methods should agree on the most important features.")


Feature    Tree imp   Perm imp  
--------------------------------
0          0.169      0.1537    
1          0.125      0.0417    
2          0.166      0.0193    
3          0.079      0.0140    
4          0.017      -0.0037   
5          0.095      0.0100    
6          0.019      -0.0007   
7          0.018      -0.0000   
8          0.175      0.0747    
9          0.139      0.0563    

Both methods should agree on the most important features.


## 17. Closed-Book Recall

Without looking back:

1. What is feature importance?
2. What is permutation importance?
3. What is a partial dependence plot?
4. Why use multiple interpretation methods?

## 18. Teach-Back Questions

Explain to another person:

- The difference between feature and permutation importance.
- How partial dependence shows feature effects.

## 19. Summary

You used feature importance, permutation importance, and partial dependence to interpret models. Interpretability builds trust and aids debugging.

## 20. Further Experiment

- Try SHAP for per-prediction explanations.
- Use LIME.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
